In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\luis\Documents\TFG - Data-Centric AI")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
from IPython.display import display

from src.phase3.config import PHASE3_DARKNESS_THRESHOLD
from utils.thresholds.quality import (
    ensure_quality_metric,
    evaluate_quality_thresholds_against_manual_labels,
    export_quality_review_images,
    get_quality_filter_spec,
    recommend_quality_threshold,
    sample_images_for_quality_threshold_review,
)

FILTER_NAME = "blur"
METADATA_PATH = PROJECT_ROOT / "data" / "phase2" / "phase2_train.csv"
IMAGES_DIR = PROJECT_ROOT / "data" / "phase2" / "frames"

REPORTS_DIR = PROJECT_ROOT / "reports" / f"{FILTER_NAME}_threshold_selection"
REVIEW_IMAGES_DIR = REPORTS_DIR / "review_images"
REVIEW_CSV_PATH = REPORTS_DIR / "manual_quality_review.csv"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

spec = get_quality_filter_spec(FILTER_NAME)
METRIC_COLUMN = spec["metric_column"]
THRESHOLD_CANDIDATES = spec["thresholds"]
N_REVIEW_IMAGES = 100
N_REVIEW_IMAGES_PER_THRESHOLD = N_REVIEW_IMAGES // len(THRESHOLD_CANDIDATES)

print("FILTER_NAME:", FILTER_NAME)
print("METADATA_PATH:", METADATA_PATH)
print("IMAGES_DIR:", IMAGES_DIR)
print("THRESHOLD_CANDIDATES:", THRESHOLD_CANDIDATES)
print("DARKNESS_PREFILTER_THRESHOLD:", PHASE3_DARKNESS_THRESHOLD)

FILTER_NAME: blur
METADATA_PATH: C:\Users\luis\Documents\TFG - Data-Centric AI\data\phase2\phase2_train.csv
IMAGES_DIR: C:\Users\luis\Documents\TFG - Data-Centric AI\data\phase2\frames
THRESHOLD_CANDIDATES: [15.0, 20.0, 25.0, 30.0, 35.0]
DARKNESS_PREFILTER_THRESHOLD: 40.0


In [3]:
df = pd.read_csv(METADATA_PATH)
df = ensure_quality_metric(
    dataframe=df,
    filter_name="darkness",
    images_dir=IMAGES_DIR,
)

images_before_darkness = len(df)
df = df[df["brightness_v_mean"] >= PHASE3_DARKNESS_THRESHOLD].copy()

df = ensure_quality_metric(
    dataframe=df,
    filter_name=FILTER_NAME,
    images_dir=IMAGES_DIR,
)

print("Images before darkness prefilter:", images_before_darkness)
print("Images after darkness prefilter:", len(df))
display(df[["filename", "brightness_v_mean", METRIC_COLUMN]].head())
display(df[METRIC_COLUMN].describe())

Images before darkness prefilter: 7978
Images after darkness prefilter: 7935


,filename,brightness_v_mean,laplacian_variance
0,20241204_134604_R0_F0_S1_8f99b423ac93fa5b.jpg,116.906393,160.175368
1,20241213_102820_R0_F0_S1_34b16ba5d8336515.jpg,84.247481,300.173878
2,20241213_102821_R2_F0_S1_34b16ba5d8336515.jpg,76.631638,137.840116
3,20241213_103001_R2_F0_S8_34b16ba5d8336515.jpg,131.618293,226.689459
4,20241213_103008_R2_F0_S9_34b16ba5d8336515.jpg,98.607903,229.116787


count     7935.000000
mean       398.721654
std        640.188669
min         10.048762
25%        137.806466
50%        227.629217
75%        393.990141
max      14615.202436
Name: laplacian_variance, dtype: float64

In [4]:
review_df = sample_images_for_quality_threshold_review(
    dataframe=df,
    filter_name=FILTER_NAME,
    thresholds=THRESHOLD_CANDIDATES,
    n_per_threshold=N_REVIEW_IMAGES_PER_THRESHOLD,
)

review_df = export_quality_review_images(
    review_df=review_df,
    images_dir=IMAGES_DIR,
    output_dir=REVIEW_IMAGES_DIR,
    image_size=(320, 180),
)

review_df.to_csv(REVIEW_CSV_PATH, index=False)

print("CSV to label:", REVIEW_CSV_PATH)
print("Images exported to:", REVIEW_IMAGES_DIR)
display(review_df.head())

CSV to label: C:\Users\luis\Documents\TFG - Data-Centric AI\reports\blur_threshold_selection\manual_quality_review.csv
Images exported to: C:\Users\luis\Documents\TFG - Data-Centric AI\reports\blur_threshold_selection\review_images


,patient_id,day,R,F,hour,histology,filename,video_filename,elapsed_seconds,crop_x,...,bbox_area_ratio,brightness_v_mean,laplacian_variance,filter_name,threshold,predicted_discard,distance_to_threshold,manual_label,manual_comment,review_image_path
0,5463981,20250414,R3,F0,NaN,Sessile_serrated_adenoma,20250414_135037_R3_F0_S1_90985f258ff468af_2.jpg,20250414_135033_R3_90985f258ff468af.mp4,4.0,410,...,0.534752,66.198326,14.128457,blur,15.0,True,0.871543,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
1,4860941,20241219,R1,F0,NaN,Sessile_serrated_adenoma,20241219_104306_R1_F0_S6_ff325cf1a4407758_3.jpg,20241219_104243_R1_ff325cf1a4407758.mp4,23.0,324,...,0.000000,64.716559,10.048762,blur,15.0,True,4.951238,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
2,5463981,20250414,R10,F0,140429.0,Sessile_serrated_adenoma,20250414_140429_R10_F0_S4_90985f258ff468af.jpg,20250414_140408_R10_90985f258ff468af.mp4,21.0,412,...,0.114300,40.656513,15.891561,blur,15.0,False,0.891561,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
3,70609365,20241219,R3,F0,92848.0,Sessile_serrated_adenoma,20241219_092848_R3_F0_S4_cf47e561beefb441.jpg,20241219_092840_R3_cf47e561beefb441.mp4,8.0,319,...,0.000000,63.999972,18.647288,blur,15.0,False,3.647288,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
4,5786206,20250430,R1,F0,NaN,Adenoma,20250430_120836_R1_F0_S2_e08bfbe5381bbc62_1.jpg,20250430_120827_R1_e08bfbe5381bbc62.mp4,9.0,351,...,0.810532,151.573595,23.182172,blur,15.0,False,8.182172,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...


In [5]:
labeled_df = pd.read_csv(REVIEW_CSV_PATH, sep=None, engine="python")
labeled_df["manual_label"] = (
    labeled_df["manual_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_labels = {"discard", "valuable", "uncertain"}
invalid_labels = set(labeled_df["manual_label"].unique()) - valid_labels

if invalid_labels:
    raise ValueError(f"Invalid labels: {invalid_labels}")

label_counts = (
    labeled_df["manual_label"]
    .value_counts()
    .rename_axis("manual_label")
    .reset_index(name="count")
)

display(label_counts)

,manual_label,count
0,valuable,68
1,discard,32


In [6]:
manual_evaluation_df = evaluate_quality_thresholds_against_manual_labels(
    labeled_df=labeled_df,
    filter_name=FILTER_NAME,
    thresholds=THRESHOLD_CANDIDATES,
    positive_labels=("discard",),
    negative_labels=("valuable",),
)

manual_evaluation_df = manual_evaluation_df.sort_values(
    ["precision", "recall", "false_positive"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(manual_evaluation_df)

,filter_name,threshold,labelled_images,true_positive,false_positive,false_negative,true_negative,precision,recall
0,blur,35.0,100,30,59,2,9,0.337079,0.93750
1,blur,25.0,100,26,53,6,15,0.329114,0.81250
2,blur,15.0,100,20,44,12,24,0.312500,0.62500
3,blur,30.0,100,26,58,6,10,0.309524,0.81250
4,blur,20.0,100,21,53,11,15,0.283784,0.65625


In [7]:
recommended_df = recommend_quality_threshold(
    evaluation_df=manual_evaluation_df,
    min_precision=0.90,
)

display(recommended_df)

selected = recommended_df.iloc[0]
FINAL_THRESHOLD = float(selected["threshold"])

print(f"Selected {FILTER_NAME} threshold: {FINAL_THRESHOLD:g}")

,filter_name,threshold,labelled_images,true_positive,false_positive,false_negative,true_negative,precision,recall
0,blur,35.0,100,30,59,2,9,0.337079,0.93750
1,blur,25.0,100,26,53,6,15,0.329114,0.81250
2,blur,15.0,100,20,44,12,24,0.312500,0.62500
3,blur,30.0,100,26,58,6,10,0.309524,0.81250
4,blur,20.0,100,21,53,11,15,0.283784,0.65625


Selected blur threshold: 35
